# text

In [2]:
# workflow: 

# - convert Hamiltonian to only clifford operations (only Pauli X operators)

In [3]:
from schwingermodel import generateSchwingerHamiltonian

In [5]:

N = 4
F = 1
Hamiltonian = generateSchwingerHamiltonian(N = N, x = (N/30)**2, lam=1000, l0= 3.1666666666, m_lat=10, g=1)

In [6]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

def bitflip_projection(op: SparsePauliOp, combine_coeffs=True) -> SparsePauliOp:
    """
    Map a SparsePauliOp to a new operator where:
        X -> X
        Y -> X
        Z -> I
        I -> I

    Parameters
    ----------
    op : SparsePauliOp
        Input Hamiltonian/operator.
    combine_coeffs : bool
        If True, simplify identical resulting Pauli strings by summing coefficients.

    Returns
    -------
    SparsePauliOp
        New operator containing only I and X.
    """
    new_terms = []

    labels = op.paulis.to_labels()
    coeffs = op.coeffs

    for label, coeff in zip(labels, coeffs):
        new_label = []
        for p in label:
            if p == 'X':
                new_label.append('X')
            elif p == 'Y':
                new_label.append('X')
            elif p == 'Z':
                new_label.append('I')
            elif p == 'I':
                new_label.append('I')
            else:
                raise ValueError(f"Unexpected Pauli character: {p}")

        new_terms.append(("".join(new_label), coeff))

    out = SparsePauliOp.from_list(new_terms)
    return out.simplify() if combine_coeffs else out

In [7]:
def apply_operator_to_state(op: SparsePauliOp, psi):
    """
    Apply SparsePauliOp to a statevector psi.

    Parameters
    ----------
    op : SparsePauliOp
    psi : array-like
        Complex statevector of length 2**N.

    Returns
    -------
    np.ndarray
        The resulting statevector.
    """
    psi = np.asarray(psi, dtype=complex)
    return op.to_matrix(sparse=False) @ psi

what bitstrings arise at all? We don't care about the phases

In [8]:
def nonzero_bitstrings(state, tol=1e-12):
    """
    Return bitstrings with nonzero amplitude in a statevector.
    """
    state = np.asarray(state)
    n = int(np.log2(len(state)))
    return [
        format(i, f"0{n}b")
        for i, amp in enumerate(state)
        if abs(amp) > tol
    ]

In [9]:
# Example parameters
N = 4
x = 1.0
lam = 0.5
l0 = 0.0
m_lat = 1.0
g = 1.0

H = generateSchwingerHamiltonian(N, x, lam, l0, m_lat, g)
H_flip = bitflip_projection(H)

print("Original Hamiltonian:")
print(H)

print("\nBit-flip projected Hamiltonian:")
print(H_flip)

Original Hamiltonian:
SparsePauliOp(['IIXX', 'IIYY', 'IXXI', 'IYYI', 'XXII', 'YYII', 'IIZZ', 'IZIZ', 'ZIIZ', 'IZZI', 'ZIZI', 'ZZII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIII'],
              coeffs=[ 0.5 +0.j,  0.5 +0.j,  0.5 +0.j,  0.5 +0.j,  0.5 +0.j,  0.5 +0.j,
  1.25+0.j,  0.75+0.j,  0.25+0.j,  0.75+0.j,  0.25+0.j,  0.25+0.j,
  2.  +0.j, -0.5 +0.j,  1.5 +0.j, -1.  +0.j,  2.5 +0.j])

Bit-flip projected Hamiltonian:
SparsePauliOp(['IIXX', 'IXXI', 'XXII', 'IIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 8.+0.j])


In [10]:
def basis_state(bitstring: str):
    n = len(bitstring)
    psi = np.zeros(2**n, dtype=complex)
    psi[int(bitstring, 2)] = 1.0
    return psi

psi0 = basis_state("0101")
psi1 = apply_operator_to_state(H_flip, psi0)

print("Resulting statevector:")
print(psi1)

print("Nonzero output bitstrings:")
print(nonzero_bitstrings(psi1))

Resulting statevector:
[0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 8.+0.j 1.+0.j 0.+0.j 0.+0.j 1.+0.j
 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
Nonzero output bitstrings:
['0011', '0101', '0110', '1001']


# This might be all we need!

In [11]:
from qiskit.quantum_info import SparsePauliOp

def bitflip_projection(op: SparsePauliOp) -> SparsePauliOp:
    terms = []
    for label, coeff in zip(op.paulis.to_labels(), op.coeffs):
        new_label = label.replace("Y", "X").replace("Z", "I")
        terms.append((new_label, coeff))
    return SparsePauliOp.from_list(terms).simplify()


def apply_ix_string_to_bitstring(ix_label: str, bitstring: str) -> str:
    out = list(bitstring)
    for j, p in enumerate(ix_label):
        if p == "X":
            out[j] = "1" if out[j] == "0" else "0"
    return "".join(out)


def reachable_bitstrings_n_steps(op: SparsePauliOp, initial_bitstrings, n_steps: int):
    """
    Ignore amplitudes entirely.
    Track only which bitstrings are reachable after n_steps applications
    of the projected operator's flip masks.
    """
    if isinstance(initial_bitstrings, str):
        current = {initial_bitstrings}
    else:
        current = set(initial_bitstrings)

    labels = op.paulis.to_labels()

    for _ in range(n_steps):
        nxt = set()
        for b in current:
            for label in labels:
                nxt.add(apply_ix_string_to_bitstring(label, b))
        current = nxt

    return current

In [25]:
test = '10' * 2

In [26]:
N = 40
H = generateSchwingerHamiltonian(N, x, lam, l0, m_lat, g)
H_flip = bitflip_projection(H)

initial_state = '01' * (N//2)

reachable = reachable_bitstrings_n_steps(H_flip, initial_state, 3)
print(len(reachable))

9920
